# Cat Podcast - Voice Generation Webhook

In [ ]:
# Cell 1: Install
!apt-get update -qq > /dev/null
!apt-get install -y ffmpeg -qq > /dev/null
!pip install flask pyngrok -qq
import os
if not os.path.exists('/content/VibeVoice/setup.py'):
    !git clone https://github.com/vibevoice-community/VibeVoice.git > /dev/null 2>&1
%cd /content/VibeVoice
!pip install -e . -qq
print('Setup done')

In [ ]:
# Cell 2: Server
import subprocess as sp
sp.run('pkill -f ngrok', shell=True, capture_output=True)
sp.run('rm -rf /tmp/ngrok*', shell=True, capture_output=True)

from flask import Flask, request, jsonify
from pyngrok import ngrok
import subprocess
import base64
import traceback

ngrok.set_auth_token('3Hg6t6KMD50TMDe7WJBy6kxICHC_34gzH8g6o9N4V7We6VDgm')

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json
        script = data.get('script', '')
        filename = data.get('filename', 'episode.txt')
        with open(filename, 'w') as f:
            f.write(script)
        os.makedirs('./outputs', exist_ok=True)
        cmd = 'python demo/inference_from_file.py --model_path microsoft/VibeVoice-1.5B --txt_path ' + filename + ' --speaker_names Frank Maya --output_dir ./outputs --cfg_scale 1.3 --device cuda'
        subprocess.run(cmd, shell=True, check=True)
        with open('./outputs/' + filename.replace('.txt', '_generated.wav'), 'rb') as f:
            audio = base64.b64encode(f.read()).decode('utf-8')
        return jsonify({'status': 'success', 'audio_base64': audio, 'sample_rate': 24000})
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e), 'trace': traceback.format_exc()}), 500

url = ngrok.connect(5000)
print('\nYOUR WEBHOOK URL: ' + str(url) + '/generate\n')
app.run(port=5000)